# Scrape DMC Records

In [ ]:
%run ../scripts/scrape_dmc.py

# Download DMC PDFs

In [ ]:
%run ../scripts/download_dmc.py

# PDF > Text Conversion test

In [ ]:
import pdfplumber

pdf_path = "../data/raw/dmc_reports/2020-05-18_15-45_Water_Level.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()

        print(text)
        print("=" * 80)

# Removing 2018/2019 PDFs

In [ ]:
from pathlib import Path
from datetime import datetime

PDF_DIR = Path("../data/raw/dmc_reports")

cutoff = datetime(2020, 1, 1)

deleted = 0
kept = 0
unrecognized = 0

for pdf in PDF_DIR.glob("*.pdf"):
    try:
        # Date is the first 10 characters of the filename
        file_date = datetime.strptime(
            pdf.name[:10],
            "%Y-%m-%d"
        )

        if file_date < cutoff or file_date == "0000-00-00":
            pdf.unlink()
            deleted += 1
        else:
            kept += 1

    except (ValueError, IndexError):
        print(f"Could not determine date: {pdf.name}")
        unrecognized += 1

print(f"Deleted: {deleted}")
print(f"Kept: {kept}")
print(f"Unrecognized: {unrecognized}")

# Seperating Water Level PDFs

In [ ]:
from pathlib import Path
import pandas as pd


PDF_DIR = Path("../data/raw/dmc_reports")

OUTPUT_FILE = Path(
    "../data/intermediate/water_level_files.csv"
)

OUTPUT_DIR = OUTPUT_FILE.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


WATER_LEVEL_KEYWORDS = [
    "water_level",
    "water level",
    "water_leval",
    "river_water_level",
    "river_water_level_report",
]


def is_water_level_file(filename):
    name = filename.lower().replace("-", "_")

    return any(
        keyword in name
        for keyword in WATER_LEVEL_KEYWORDS
    )


def main():

    rows = []

    for pdf in PDF_DIR.glob("*.pdf"):

        if is_water_level_file(pdf.name):

            rows.append({
                "filename": pdf.name,
                "filepath": str(pdf),
            })

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=["filename"]
    )

    df = df.sort_values(
        "filename"
    )

    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

    print(
        f"Found {len(df)} possible water-level PDFs"
    )

    print(
        f"Saved to: {OUTPUT_FILE}"
    )


if __name__ == "__main__":
    main()

In [ ]:
%run ../scripts/build_water_level_dataset.py